In [17]:
SEED = 42

In [18]:
from pathlib import Path
from helpers.data.ticker_loader import load_tickers

TICKERS_FILE = Path("tickers.txt")
if not TICKERS_FILE.exists():
    TICKERS_FILE = Path("NN_Trading_project/tickers.txt")

# None        → all tickers
# ["AAPL", …] → explicit list
# 42          → random sample of 42 (seeded by SEED)
TICKER_SUBSET = 2

TICKERS = load_tickers(TICKERS_FILE, subset=TICKER_SUBSET, seed=SEED)
print(f"Using {len(TICKERS)} tickers")


Using 2 tickers


In [19]:
import pandas as pd
import datetime

# ============================================================
# DATE SETTINGS
# ============================================================

# --- Date Range ---
INTERVAL   = "1d"
START_DATE = pd.Timestamp("2025-01-01")           # inclusive
END_DATE   = pd.Timestamp(datetime.date.today())  # inclusive

# --- Train / Val / Test Split ---
# Train: Date <= TRAIN_END_DATE
# Val:   (TRAIN_END_DATE, VAL_END_DATE]
# Test:  Date > VAL_END_DATE
TRAIN_END_DATE = pd.Timestamp("2025-11-30")
VAL_END_DATE   = pd.Timestamp("2026-01-20")

# --- Exclusion Window (applied during featurize) ---
EXCLUDE_START_DATE = "2018-09-01"  # inclusive
EXCLUDE_END_DATE   = "2023-06-30"  # inclusive


# --- Exclusion Window (Features) ---
REBUILD_FEATURE_CACHE = True  # set True to recompute when features/WINDOW change


In [20]:
# ============================================================
# HYPERPARAMETERS - MODEL & TRAINING
# ============================================================

# --- Trading / Labeling ---
# Buy at market OPEN using prior day's OHLCV + current day's OPEN only.
HORIZON_BARS     = 0         # look-ahead bars for profit hit
PROFIT_THRESHOLD = 1 / 100   # 1% profit target vs current day's OPEN
STOP_LOSS        = -2 / 100  # -2% stop loss vs current day's OPEN
WINDOW           = 30        # look-back window for features

# --- Training ---
MAX_EPOCHS    = 50
PATIENCE      = 10   # early-stopping patience

BUY_THRESHOLD = 0.5  # minimum probability to classify as BUY

# --- Plotting ---
PLOT_GRAPH1_EPOCH_INTERVAL = 1
PLOT_GRAPH2_BATCH_INTERVAL = 5
PLOT_GRAPH3_EPOCH_INTERVAL = 1

# --- Optimizer ---
BATCH_SIZE   = 64

# --- Model Architecture ---
HIDDEN_SIZES = [64]  # e.g. [1024, 512] or [256, 128]

# --- Data / DataLoader ---
SPLIT_FRAC  = 0.85  # train fraction (time-based)
NUM_WORKERS = 16    # DataLoader workers (0 for debugging)


In [21]:
from pathlib import Path
from helpers.data.date_config_manager import check_and_refresh_date_config

_current_config = {
    "INTERVAL":           str(INTERVAL),
    "START_DATE":         str(START_DATE.date()),
    "END_DATE":           str(END_DATE.date()),
    "TRAIN_END_DATE":     str(TRAIN_END_DATE.date()),
    "VAL_END_DATE":       str(VAL_END_DATE.date()),
    "EXCLUDE_START_DATE": str(EXCLUDE_START_DATE),
    "EXCLUDE_END_DATE":   str(EXCLUDE_END_DATE),
    "TICKER_SUBSET":      str(TICKER_SUBSET),
}

check_and_refresh_date_config(
    current_config = _current_config,
    config_path    = Path.cwd() / "date_config.txt",
    stocks_dir     = Path.cwd() / "dataset" / "stocks",
)


Date config changed — clearing cached CSVs for a fresh download.
  Previous config:
    INTERVAL: 1d
    START_DATE: 2025-01-01
    END_DATE: 2026-02-27
    TRAIN_END_DATE: 2025-11-30
    VAL_END_DATE: 2026-01-20
    EXCLUDE_START_DATE: 2018-09-01
    EXCLUDE_END_DATE: 2023-06-30
    TICKER_SUBSET: 5 <-- CHANGED
  Deleted: c:\Users\adham\OneDrive\Desktop\NN_Trading_Bot_Private\NN_Trading_project\dataset\stocks
  Updated: c:\Users\adham\OneDrive\Desktop\NN_Trading_Bot_Private\NN_Trading_project\date_config.txt


True

In [22]:
# Step 1: download NASDAQ data into the *dataset* directory using yfinance (DAILY)
from pathlib import Path
import pandas as pd
from helpers.data.data_downloader import download_tickers

# Root directory where yfinance CSVs will be stored

data_root = Path.cwd() / "dataset"
stocks_dir = data_root / "stocks"

# call helper
_summary = download_tickers(
    tickers= TICKERS,
    start=   START_DATE,
    end=     END_DATE,
    interval=INTERVAL,
    out_dir= stocks_dir,
)


 OK   :: BSX rows=288 | downloaded 1/2
 OK   :: SRPT rows=288 | downloaded 2/2
Path to dataset files: c:\Users\adham\OneDrive\Desktop\NN_Trading_Bot_Private\NN_Trading_project\dataset
Individual ticker CSVs are in: c:\Users\adham\OneDrive\Desktop\NN_Trading_Bot_Private\NN_Trading_project\dataset\stocks
Total number of rows across all CSVs on disk: 578
Newly downloaded rows in this run: 576
Downloaded tickers in this run: 2/2

All requested tickers that were downloaded in this run succeeded.


# Building Features + Labels

In [24]:
import pickle
import shutil
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from helpers.feature.feature_builder import precompute_and_cache, FEATURE_COLS
from helpers.data.dataset import StockDatasetSafe, is_cache_valid

# ── Paths ────────────────────────────────────────────────────────────────────
root       = Path.cwd() / "dataset"
stocks_dir = root / "stocks"
assert stocks_dir.exists(), f"Missing: {stocks_dir}"

files     = sorted(stocks_dir.glob("*.csv"))
cache_dir = Path.cwd() / f".feature_cache_forward_return_w{WINDOW}"
cache_dir.mkdir(parents=True, exist_ok=True)
scaler_path = cache_dir / "scaler.pkl"
index_path  = cache_dir / "index.pkl"

# ── Clear old cache if requested ─────────────────────────────────────────────
if REBUILD_FEATURE_CACHE:
    stale = [p for p in Path.cwd().glob(".feature*") if p.exists()]
    for p in stale:
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
        else:
            p.unlink(missing_ok=True)
        print(f"[Cache] Removed: {p}")
    if not stale:
        print("[Cache] No stale .feature* paths found.")
    time.sleep(1)

# ── Build or load cache ───────────────────────────────────────────────────────
# Validates that scaler, index, AND every referenced .npz file all exist.
cache_ready = is_cache_valid(scaler_path, index_path)
if not cache_ready and cache_dir.exists():
    print("[Cache] Stale / incomplete cache detected — wiping cache dir.")
    shutil.rmtree(cache_dir)

# Ensure cache_dir exists before building or loading
cache_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_FEATURE_CACHE or not cache_ready:
    scaler, index = precompute_and_cache(
        files           = files,
        window          = WINDOW,
        cache_dir       = cache_dir,
        scaler_path     = scaler_path,
        index_path      = index_path,
        horizon_bars    = HORIZON_BARS,
        train_end_date  = TRAIN_END_DATE,
        val_end_date    = VAL_END_DATE,
        profit_threshold= PROFIT_THRESHOLD,
        stop_loss       = STOP_LOSS,
        exclude_start   = EXCLUDE_START_DATE,
        exclude_end     = EXCLUDE_END_DATE,
    )
else:
    print("[Cache] Using existing feature cache.")
    with open(scaler_path, "rb") as f: scaler = pickle.load(f)
    with open(index_path,  "rb") as f: index  = pickle.load(f)

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds = StockDatasetSafe(index, scaler, "train")
val_ds   = StockDatasetSafe(index, scaler, "val")
test_ds  = StockDatasetSafe(index, scaler, "test")

# ── DataLoaders ───────────────────────────────────────────────────────────────
_pin    = torch.cuda.is_available()
_kwargs = dict(
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    pin_memory  = _pin,
    persistent_workers = (NUM_WORKERS > 0),
    prefetch_factor    = 2 if NUM_WORKERS > 0 else None,
)

train_loader = DataLoader(train_ds, shuffle=True,  **_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_kwargs)

# ── Sanity check ──────────────────────────────────────────────────────────────
xb, yb = next(iter(train_loader))
print(f"Train samples : {len(train_ds):,}")
print(f"Val   samples : {len(val_ds):,}")
print(f"Test  samples : {len(test_ds):,}")
print(f"X batch : {xb.shape}  {xb.dtype}")
print(f"y batch : {yb.shape}  {yb.dtype}")
print(f"Feature count : {len(FEATURE_COLS)} base + {WINDOW-1} lags = {len(FEATURE_COLS) * WINDOW}")

[Cache] Removed: c:\Users\adham\OneDrive\Desktop\NN_Trading_Bot_Private\NN_Trading_project\.feature_cache_forward_return_w30
[Cache] Precomputing features (leakage-safe splits)...
[Cache] (1/2) BSX.csv
[Cache] (2/2) SRPT.csv
[Cache] Done — used=2, skipped=0


C:\Users\adham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


KeyboardInterrupt: 

# XGBoost

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss

# ============================================================
# Helpers
# ============================================================

def loader_to_numpy(loader):
    """Flatten a torch DataLoader into numpy arrays (order depends on loader shuffle!)."""
    X_list, y_list = [], []
    for xb, yb in loader:
        X_list.append(xb.numpy())
        y_list.append(yb.numpy())
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X, y


def predict_probs_booster(booster: xgb.Booster, X: np.ndarray, best_ntree: int) -> np.ndarray:
    """Predict probabilities from a native xgboost Booster (version-safe)."""
    dmat = xgb.DMatrix(X)
    try:
        probs = booster.predict(dmat, iteration_range=(0, best_ntree))
    except TypeError:
        probs = booster.predict(dmat, ntree_limit=best_ntree)
    return probs.astype(np.float32)


def buy_metrics(y_true, probs, threshold: float):
    """Compute confusion + accuracy + P(success | BUY)."""
    probs = np.asarray(probs).ravel()
    y_true = np.asarray(y_true).ravel().astype(np.int64)
    pred = (probs >= threshold).astype(np.int64)

    tp = int(((pred == 1) & (y_true == 1)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())

    total = max(1, len(y_true))
    acc = 100.0 * (tp + tn) / total

    buy_total = max(1, tp + fp)
    buy_success = 100.0 * tp / buy_total

    return {
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "acc": acc,
        "buy_success": buy_success
    }


def evaluate_and_save_test(
    booster: xgb.Booster,
    best_ntree: int,
    X_test: np.ndarray,
    y_test: np.ndarray,
    prob_threshold: float,
    save_csv: str = "test_predictions_full.csv",
    test_tickers: list = None,
    pct_changes: list = None,
):
    """Run full test eval, save predictions CSV, and plot charts."""
    probs = predict_probs_booster(booster, X_test, best_ntree)
    pred = (probs >= prob_threshold).astype(np.int64)
    ytrue = y_test.astype(np.int64)
    correct = (pred == ytrue)

    df = pd.DataFrame({
        "idx": np.arange(len(ytrue), dtype=np.int64),
        "prob_buy": probs.astype(np.float32),
        "pred": pred,
        "actual": ytrue,
        "correct": correct,
        "decision": np.where(pred == 1, "BUY", "NO-BUY"),
        "actual_label": np.where(ytrue == 1, "BUY", "NO-BUY"),
    })

    if test_tickers is not None and len(test_tickers) == len(df):
        df["ticker"] = test_tickers
    if pct_changes is not None and len(pct_changes) == len(df):
        df["pct_change"] = pct_changes

    df.to_csv(save_csv, index=False)
    print(f"Saved full test predictions to: {save_csv}")

    stats = buy_metrics(ytrue, probs, prob_threshold)
    tp, fp, tn, fn = stats["tp"], stats["fp"], stats["tn"], stats["fn"]
    acc = stats["acc"]
    pct_buy_success = stats["buy_success"]

    print(f">>>>>>>>> Percentage of successful BUYs: {pct_buy_success:.2f}% <<<<<<<<<<")

    summary = pd.DataFrame({
        "metric": [
            "total",
            "accuracy_%",
            "successful_BUY(TP)",
            "failed_BUY(FP)",
            "successful_NO_BUY(TN)",
            "failed_NO_BUY(FN)",
        ],
        "value": [len(ytrue), acc, tp, fp, tn, fn],
    })
    display(summary)

    total = max(1, len(ytrue))
    pct_tp_all = 100.0 * tp / total
    pct_fp_all = 100.0 * fp / total
    pct_tn_all = 100.0 * tn / total
    pct_fn_all = 100.0 * fn / total

    buy_total = max(1, tp + fp)
    pct_buy_success = 100.0 * tp / buy_total
    pct_buy_fail = 100.0 * fp / buy_total

    nobuy_total = max(1, tn + fn)
    pct_nobuy_success = 100.0 * tn / nobuy_total
    pct_nobuy_fail = 100.0 * fn / nobuy_total

    summary_pct = pd.DataFrame({
        "category": [
            "BUY_success_TP",
            "BUY_fail_FP",
            "NO_BUY_success_TN",
            "NO_BUY_fail_FN",
        ],
        "count": [tp, fp, tn, fn],
        "percent_of_all_%": [pct_tp_all, pct_fp_all, pct_tn_all, pct_fn_all],
        "percent_given_decision_%": [
            pct_buy_success,
            pct_buy_fail,
            pct_nobuy_success,
            pct_nobuy_fail,
        ],
    })
    display(summary_pct)

    # Chart 1: BUY vs NO-BUY (stacked success/fail)
    labels = ["BUY", "NO-BUY"]
    success = np.array([tp, tn], dtype=np.int64)
    fail    = np.array([fp, fn], dtype=np.int64)
    x = np.arange(len(labels))

    plt.figure(figsize=(7, 4))
    plt.bar(x, success, label="successful")
    plt.bar(x, fail, bottom=success, label="failed")
    plt.xticks(x, labels)
    plt.ylabel("count")
    plt.title(f"Test decisions @ thr={prob_threshold} | acc={acc:.2f}% | N={len(ytrue)}")
    plt.grid(True, alpha=0.2, axis="y")
    plt.legend()
    plt.show()

    # Chart 2: 4-way outcomes
    four_labels = ["BUY success (TP)", "BUY fail (FP)", "NO-BUY success (TN)", "NO-BUY fail (FN)"]
    four_counts = [tp, fp, tn, fn]
    x2 = np.arange(len(four_labels))

    plt.figure(figsize=(9, 4))
    plt.bar(x2, four_counts)
    plt.xticks(x2, four_labels, rotation=20, ha="right")
    plt.ylabel("count")
    plt.title(f"BUY / NO-BUY outcomes @ thr={prob_threshold} | N={len(ytrue)}")
    plt.grid(True, alpha=0.2, axis="y")

    for i, c in enumerate(four_counts):
        plt.text(i, c + max(1, len(ytrue)) * 0.01, str(c), ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    plt.show()

    return df, summary, summary_pct


def compute_logloss_curve_over_iters(
    booster: xgb.Booster,
    X: np.ndarray,
    y: np.ndarray,
    iters: np.ndarray,
) -> np.ndarray:
    """Compute logloss at a set of iteration counts (post-training)."""
    dmat = xgb.DMatrix(X)
    y = y.astype(np.int64)
    out = []
    for k in iters:
        try:
            p = booster.predict(dmat, iteration_range=(0, int(k)))
        except TypeError:
            p = booster.predict(dmat, ntree_limit=int(k))
        out.append(log_loss(y, p))
    return np.asarray(out, dtype=np.float64)


# ============================================================
# 1) Build arrays from loaders (already scaled in your pipeline)
# NOTE: val_loader/test_loader should be shuffle=False to preserve order.
# ============================================================

X_train, y_train = loader_to_numpy(train_loader)
X_val,   y_val   = loader_to_numpy(val_loader)
X_test,  y_test  = loader_to_numpy(test_loader)

print(f"Train: {X_train.shape} pos={int((y_train==1).sum())} neg={int((y_train==0).sum())}")
print(f"Val:   {X_val.shape}   pos={int((y_val==1).sum())} neg={int((y_val==0).sum())}")
print(f"Test:  {X_test.shape}  pos={int((y_test==1).sum())} neg={int((y_test==0).sum())}")

# class imbalance
num_pos = float((y_train == 1).sum())
num_neg = float((y_train == 0).sum())
scale_pos_weight = num_neg / max(1.0, num_pos)
print(f"scale_pos_weight (neg/pos) = {scale_pos_weight:.4f}")

# ============================================================
# 2) Train XGBoost (native API) with early stopping + LOGLOSS CURVES
# ============================================================

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

params = {
    "max_depth": 3,
    "eta": 0.01, # LR 
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "alpha": 3.0,
    "tree_method": "hist",
    "scale_pos_weight": scale_pos_weight,
    "lambda" : 2.0

}

PATIENCE = 100
num_boost_round = 10000 # max rounds
es_rounds = max(5, PATIENCE * 5)  # e.g. 15

evals_result = {}  # <-- collect per-iteration train/val metrics

print(f"Training with early stopping: rounds={num_boost_round}, early_stopping_rounds={es_rounds}")
booster = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtrain, "train"), (dval, "val")],  # keep val LAST so early stopping uses val
    early_stopping_rounds=es_rounds,
    evals_result=evals_result,
    verbose_eval=True,
)

best_ntree = int(booster.best_iteration + 1) if booster.best_iteration is not None else num_boost_round
print(f"Best iteration used: {best_ntree}")

# quick train/val logloss at best_ntree
probs_train = predict_probs_booster(booster, X_train, best_ntree)
probs_val   = predict_probs_booster(booster, X_val,   best_ntree)

print(f"Train logloss: {log_loss(y_train, probs_train):.6f}")
print(f"Val   logloss: {log_loss(y_val,   probs_val):.6f}")

train_stats = buy_metrics(y_train, probs_train, BUY_THRESHOLD)
val_stats   = buy_metrics(y_val,   probs_val,   BUY_THRESHOLD)

print(f"Train: acc={train_stats['acc']:.2f}% | P(success|BUY)={train_stats['buy_success']:.2f}% "
      f"| TP={train_stats['tp']} FP={train_stats['fp']} TN={train_stats['tn']} FN={train_stats['fn']}")
print(f"Val:   acc={val_stats['acc']:.2f}% | P(success|BUY)={val_stats['buy_success']:.2f}% "
      f"| TP={val_stats['tp']} FP={val_stats['fp']} TN={val_stats['tn']} FN={val_stats['fn']}")

# ============================================================
# 2b) NEW: Plot Train / Val / Test logloss as training progresses
#   - Train/Val from evals_result (free)
#   - Test computed post-training (sampled to keep it fast)
# ============================================================

train_ll = np.asarray(evals_result.get("train", {}).get("logloss", []), dtype=np.float64)
val_ll   = np.asarray(evals_result.get("val", {}).get("logloss", []),   dtype=np.float64)

iters_full = np.arange(1, len(train_ll) + 1)

# Sample test curve to avoid O(num_rounds * N_test) being too heavy
max_points = 200
step = max(1, int(np.ceil(best_ntree / max_points)))
iters_test = np.arange(1, best_ntree + 1, step)
if iters_test[-1] != best_ntree:
    iters_test = np.append(iters_test, best_ntree)

test_ll = compute_logloss_curve_over_iters(booster, X_test, y_test, iters_test)

plt.figure(figsize=(10, 5))
if len(train_ll) > 0:
    plt.plot(iters_full, train_ll, label="train logloss")
if len(val_ll) > 0:
    plt.plot(iters_full, val_ll, label="val logloss")
plt.plot(iters_test, test_ll, marker="o", linestyle="-", label="test logloss (sampled)")

plt.axvline(best_ntree, linestyle="--", label=f"best_ntree={best_ntree}")
plt.xlabel("Boosting iteration")
plt.ylabel("Logloss")
plt.title("Train vs Val vs Test Logloss Over Boosting Iterations")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

# ============================================================
# 3) Save bundle (THIS is why later you must NOT call predict_proba)
# ============================================================

bundle = {"booster": booster, "best_ntree": best_ntree}
joblib.dump(bundle, "best_model_xgb.pkl")
print("Saved XGBoost booster bundle to best_model_xgb.pkl")


### Evaluate on Test Data 

In [ ]:
# ============================================================
# 4) Evaluate on test + save predictions + charts (FIXED)
#   - supports new index keys: test_start (preferred), val_cut (legacy)
#   - aligns pct_change by cached decision dates (no iloc misalignment)
# ============================================================

import pickle
with open(index_path, "rb") as f:
    index = pickle.load(f)

test_tickers = []
pct_changes  = []

for entry in index:
    data = np.load(entry["cache_file"])
    ticker = entry["ticker"]

    # New leakage-safe key is "test_start". Keep backward compat with "val_cut".
    test_start = entry.get("test_start", entry.get("val_cut", None))
    if test_start is None:
        raise KeyError(f"Index entry for {ticker} missing both 'test_start' and 'val_cut'.")

    test_start = int(test_start)

    y_full = data["y"]
    dates_full = data.get("dates", None)

    n_total = int(len(y_full))
    n_test = n_total - test_start
    if n_test <= 0:
        continue

    test_tickers.extend([ticker] * n_test)

    # ----- pct_change aligned by DATE (preferred) -----
    # pct_change here = (Close - Open) / Open * 100 for the same decision date.
    # If you want a different return definition, change it here.
    if dates_full is None:
        # No dates stored -> can't align safely; fill NaNs
        pct_changes.extend([np.nan] * n_test)
        continue

    test_dates = pd.to_datetime(dates_full[test_start:]).tz_localize(None)

    csv_path = stocks_dir / f"{ticker}.csv"
    try:
        df_csv = pd.read_csv(csv_path)
        if "Date" not in df_csv.columns:
            pct_changes.extend([np.nan] * n_test)
            continue

        df_csv["Date"] = pd.to_datetime(df_csv["Date"], errors="coerce")
        df_csv = df_csv.dropna(subset=["Date"]).sort_values("Date")
        df_csv = df_csv.set_index("Date")
        df_csv = df_csv[~df_csv.index.duplicated(keep="last")]

        # Reindex to the cached feature/label decision dates
        sub = df_csv.reindex(test_dates)

        # Ensure numeric
        sub["Open"]  = pd.to_numeric(sub.get("Open"), errors="coerce")
        sub["Close"] = pd.to_numeric(sub.get("Close"), errors="coerce")

        pct = ((sub["Close"] - sub["Open"]) / sub["Open"]) * 100.0
        pct = pct.to_numpy(dtype=np.float64)

        # Make sure we add exactly n_test values
        if len(pct) != n_test:
            # pad/truncate defensively
            if len(pct) < n_test:
                pct = np.concatenate([pct, np.full(n_test - len(pct), np.nan)])
            else:
                pct = pct[:n_test]

        pct_changes.extend(pct.tolist())

    except Exception:
        pct_changes.extend([np.nan] * n_test)

# Sanity: these should match X_test length
assert len(test_tickers) == len(X_test), f"test_tickers len mismatch: {len(test_tickers)} vs X_test {len(X_test)}"
assert len(pct_changes)  == len(X_test), f"pct_changes len mismatch: {len(pct_changes)} vs X_test {len(X_test)}"

df_test_preds, df_test_summary, df_test_summary_pct = evaluate_and_save_test(
    booster=booster,
    best_ntree=best_ntree,
    X_test=X_test,
    y_test=y_test,
    prob_threshold=BUY_THRESHOLD,
    save_csv="test_predictions_full.csv",
    test_tickers=test_tickers,
    pct_changes=pct_changes,
)

# Tune Buy Threthold

In [ ]:
# ============================================================
# Tune BUY_THRESHOLD on Validation Set
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("TUNING BUY_THRESHOLD ON VALIDATION SET")
print("=" * 80)

# Get predictions on validation set
probs_val_tune = predict_probs_booster(booster, X_val, best_ntree)

# Define cost model (based on your PROFIT_THRESHOLD and STOP_LOSS_THRESHOLD)
# Assuming typical trading costs and slippage
TRADING_COST = 0.001  # 0.1% per round trip (buy + sell)

# Test different thresholds
threshold_values = np.arange(0.45, 0.85, 0.01)
results = []

for threshold in threshold_values:
    pred = (probs_val_tune >= threshold).astype(np.int64)
    
    # Count trades and outcomes
    tp = int(((pred == 1) & (y_val == 1)).sum())  # True Positive (BUY and profitable)
    fp = int(((pred == 1) & (y_val == 0)).sum())  # False Positive (BUY but not profitable)
    
    total_trades = tp + fp
    
    if total_trades == 0:
        continue
        
    win_rate = 100.0 * tp / total_trades
    
    # Calculate expected returns
    # When TP: we hit profit target = PROFIT_THRESHOLD gain - costs
    # When FP: we don't hit profit (could hit stop loss or just no profit)
    # For simplicity, assume FP means 0% return on average after costs
    
    avg_win = PROFIT_THRESHOLD - TRADING_COST  # profit minus costs
    avg_loss = -TRADING_COST  # just costs, no gain
    
    # Expected value per trade
    ev_per_trade = (tp / total_trades) * avg_win + (fp / total_trades) * avg_loss
    
    # Trades per day (assuming validation set spans multiple days)
    # We need to know how many days in validation set
    # Approximate: if val set is ~15% of data, and total data is from July to now (~6-7 months)
    # That's roughly 180-210 days total, so val set is ~27-31 days
    # For safer estimate, let's use 30 days
    val_days = 30  # approximate
    trades_per_day = total_trades / val_days
    
    # Calculate % of BUY trades out of all samples
    total_samples = len(y_val)
    buy_pct = 100.0 * total_trades / total_samples
    
    results.append({
        'threshold': threshold,
        'total_trades': total_trades,
        'trades_per_day': trades_per_day,
        'buy_%': buy_pct,
        'win_rate_%': win_rate,
        'tp': tp,
        'fp': fp,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'ev_per_trade': ev_per_trade,
        'p_success_after_costs': win_rate  # Win rate already accounts for hitting profit target
    })

df_threshold_tune = pd.DataFrame(results)

# Find optimal threshold (maximize EV per trade)
optimal_idx = df_threshold_tune['ev_per_trade'].idxmax()
optimal_threshold = df_threshold_tune.loc[optimal_idx, 'threshold']
optimal_ev = df_threshold_tune.loc[optimal_idx, 'ev_per_trade']

print(f"\n🎯 OPTIMAL THRESHOLD: {optimal_threshold:.3f}")
print(f"   Expected Value per Trade: {optimal_ev:.4f} ({optimal_ev*100:.2f}%)")
print("=" * 80)

# Display metrics at optimal threshold
optimal_row = df_threshold_tune.loc[optimal_idx]
print(f"\nMETRICS AT OPTIMAL THRESHOLD ({optimal_threshold:.3f}):")
print("-" * 80)
print(f"  Trades per day:           {optimal_row['trades_per_day']:.2f}")
print(f"  % BUY (out of all samples): {optimal_row['buy_%']:.2f}%")
print(f"  Win rate (TP hit %):      {optimal_row['win_rate_%']:.2f}%")
print(f"  Average win:              {optimal_row['avg_win']*100:.2f}%")
print(f"  Average loss:             {optimal_row['avg_loss']*100:.2f}%")
print(f"  Expected value per trade: {optimal_row['ev_per_trade']*100:.2f}%")
print(f"  P(success | BUY) after costs: {optimal_row['p_success_after_costs']:.2f}%")
print(f"  Total trades (val set):   {optimal_row['total_trades']:.0f}")
print(f"  True Positives (TP):      {optimal_row['tp']:.0f}")
print(f"  False Positives (FP):     {optimal_row['fp']:.0f}")
print("=" * 80)

# Save tuning results to CSV
df_threshold_tune.to_csv('threshold_tuning_results.csv', index=False)
print(f"\n✓ Saved threshold tuning results to: threshold_tuning_results.csv")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: EV per trade vs threshold
ax1 = axes[0, 0]
ax1.plot(df_threshold_tune['threshold'], df_threshold_tune['ev_per_trade']*100, 'b-', linewidth=2)
ax1.axvline(optimal_threshold, color='r', linestyle='--', linewidth=2, label=f'Optimal: {optimal_threshold:.3f}')
ax1.axhline(0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
ax1.set_xlabel('BUY_THRESHOLD', fontsize=12)
ax1.set_ylabel('Expected Value per Trade (%)', fontsize=12)
ax1.set_title('Expected Value vs Threshold', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot 2: Win rate vs threshold
ax2 = axes[0, 1]
ax2.plot(df_threshold_tune['threshold'], df_threshold_tune['win_rate_%'], 'g-', linewidth=2)
ax2.axvline(optimal_threshold, color='r', linestyle='--', linewidth=2, label=f'Optimal: {optimal_threshold:.3f}')
ax2.set_xlabel('BUY_THRESHOLD', fontsize=12)
ax2.set_ylabel('Win Rate (%)', fontsize=12)
ax2.set_title('Win Rate vs Threshold', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Plot 3: Trading frequency as % of all samples vs threshold
ax3 = axes[1, 0]
ax3.plot(df_threshold_tune['threshold'], df_threshold_tune['buy_%'], 'purple', linewidth=2)
ax3.axvline(optimal_threshold, color='r', linestyle='--', linewidth=2, label=f'Optimal: {optimal_threshold:.3f}')
ax3.set_xlabel('BUY_THRESHOLD', fontsize=12)
ax3.set_ylabel('% of Total Samples', fontsize=12)
ax3.set_title('Trading Frequency (% BUY) vs Threshold', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

# Plot 4: Total trades vs threshold
ax4 = axes[1, 1]
ax4.plot(df_threshold_tune['threshold'], df_threshold_tune['total_trades'], 'orange', linewidth=2)
ax4.axvline(optimal_threshold, color='r', linestyle='--', linewidth=2, label=f'Optimal: {optimal_threshold:.3f}')
ax4.set_xlabel('BUY_THRESHOLD', fontsize=12)
ax4.set_ylabel('Total Trades (Validation Set)', fontsize=12)
ax4.set_title('Total Trades vs Threshold', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()

plt.tight_layout()
plt.show()

# Create comprehensive summary table
print("\n" + "=" * 80)
print("COMPREHENSIVE THRESHOLD ANALYSIS")
print("=" * 80)
print("\nTop 10 Thresholds by Expected Value:")
top_10_by_ev = df_threshold_tune.nlargest(10, 'ev_per_trade')[['threshold', 'trades_per_day', 'buy_%', 'win_rate_%', 'ev_per_trade', 'total_trades']]
top_10_by_ev['ev_per_trade_%'] = top_10_by_ev['ev_per_trade'] * 100
top_10_by_ev = top_10_by_ev.drop('ev_per_trade', axis=1)

# Save top 10 table to CSV
top_10_by_ev.to_csv('top_10_thresholds_by_ev.csv', index=False)
print(f"✓ Saved top 10 thresholds to: top_10_thresholds_by_ev.csv\n")

display(top_10_by_ev)

print("\n" + "=" * 80)
print(f"💡 RECOMMENDATION: Update BUY_THRESHOLD to {optimal_threshold:.3f} for optimal performance")
print("=" * 80)

# Eval Data Analysis

In [ ]:
# ============================================================
# Trades-per-day table on VALIDATION set @ selected threshold
# Includes: num trades, % success, % fail
# Plot: green=success trades, red=fail trades (stacked)
# ============================================================

import numpy as np
import pandas as pd
import pickle
import joblib
import xgboost as xgb
import matplotlib.pyplot as plt

# ----------------------------
# CONFIG
# ----------------------------
SELECTED_THRESHOLD = 0.9  # <-- set your threshold here (e.g. optimal_threshold)

# ----------------------------
# Load cache index + scaler
# ----------------------------
with open(index_path, "rb") as f:
    index = pickle.load(f)

with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

# ----------------------------
# Load model bundle (booster + best_ntree)
# ----------------------------
bundle = joblib.load("best_model_xgb.pkl")
booster = bundle["booster"]
best_ntree = int(bundle["best_ntree"])

def predict_probs_booster(booster: xgb.Booster, X: np.ndarray, best_ntree: int) -> np.ndarray:
    """Predict probabilities from a native xgboost Booster (version-safe)."""
    dmat = xgb.DMatrix(X)
    try:
        probs = booster.predict(dmat, iteration_range=(0, best_ntree))
    except TypeError:
        probs = booster.predict(dmat, ntree_limit=best_ntree)
    return probs.astype(np.float32)

# ----------------------------
# Build validation predictions dataframe WITH DATES + TICKER
# ----------------------------
rows = []

for entry in index:
    data = np.load(entry["cache_file"])
    ticker = entry["ticker"]

    X_full = data["X"].astype(np.float32)
    y_full = data["y"].astype(np.float32)
    dates  = pd.to_datetime(data["dates"])

    val_start = int(entry["val_start"])
    val_end   = int(entry["val_end"])

    # Validation slice
    Xv = X_full[val_start:val_end]
    yv = y_full[val_start:val_end]
    dv = dates[val_start:val_end]

    if len(Xv) == 0:
        continue

    # Scale like training pipeline
    Xv_scaled = scaler.transform(Xv).astype(np.float32, copy=False)

    # Predict
    pv = predict_probs_booster(booster, Xv_scaled, best_ntree)

    rows.append(pd.DataFrame({
        "Date": dv,
        "ticker": ticker,
        "y_true": yv.astype(np.int8),
        "prob_buy": pv,
    }))

val_preds = pd.concat(rows, ignore_index=True)

# Normalize Date to pure date (no time)
val_preds["Date"] = pd.to_datetime(val_preds["Date"]).dt.normalize()

# ✅ Total number of validation days (unique dates in val across all tickers)
num_val_days = int(val_preds["Date"].nunique())

# Decision at threshold
val_preds["pred_buy"] = (val_preds["prob_buy"] >= SELECTED_THRESHOLD).astype(np.int8)

# Consider only executed trades (pred_buy == 1)
trades = val_preds[val_preds["pred_buy"] == 1].copy()

# success = label == 1, fail = label == 0
trades["is_success"] = (trades["y_true"] == 1).astype(np.int8)
trades["is_fail"]    = (trades["y_true"] == 0).astype(np.int8)

# ----------------------------
# ✅ Trades-per-day table with success/fail
# ----------------------------
daily = trades.groupby("Date").agg(
    num_trades=("pred_buy", "size"),
    num_success=("is_success", "sum"),
    num_fail=("is_fail", "sum"),
).reset_index().sort_values("Date")

daily["pct_success"] = np.where(daily["num_trades"] > 0, 100.0 * daily["num_success"] / daily["num_trades"], np.nan)
daily["pct_fail"]    = np.where(daily["num_trades"] > 0, 100.0 * daily["num_fail"] / daily["num_trades"], np.nan)

# Clean column order for display
daily_table = daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]].copy()

display(daily_table)

print(f"\nThreshold: {SELECTED_THRESHOLD:.3f}")
print(f"Validation days (unique dates): {num_val_days}")
print(f"Total BUY trades in val: {int(daily['num_trades'].sum())}")
print(f"Days with >=1 trade: {len(daily)}")
print(f"Overall P(success | BUY) on val @ thr={SELECTED_THRESHOLD:.3f}: "
      f"{100.0 * daily['num_success'].sum() / max(1, daily['num_trades'].sum()):.2f}%")

# ----------------------------
# Plot: stacked bars (green=success, red=fail)
# ----------------------------
x_labels = daily["Date"].dt.strftime("%Y-%m-%d").to_list()
x = np.arange(len(x_labels))

plt.figure(figsize=(14, 6))
plt.bar(x, daily["num_success"].to_numpy(), color="green", label="success (TP)")
plt.bar(x, daily["num_fail"].to_numpy(), bottom=daily["num_success"].to_numpy(), color="red", label="fail (FP)")

plt.xticks(x, x_labels, rotation=45, ha="right")
plt.ylabel("Number of BUY trades")
plt.title(f"Validation BUY trades per day @ thr={SELECTED_THRESHOLD:.3f} (green=success, red=fail)")
plt.grid(True, alpha=0.2, axis="y")
plt.legend()
plt.tight_layout()
plt.show()


# Test Data Analysis

In [ ]:
# ============================================================
# Trades-per-day table on TEST set @ selected threshold
# Includes: num trades, % success, % fail
# Plot: green=success trades, red=fail trades (stacked)
# ============================================================

import numpy as np
import pandas as pd
import pickle
import joblib
import xgboost as xgb
import matplotlib.pyplot as plt

# ----------------------------
# CONFIG
# ----------------------------
SELECTED_THRESHOLD = 0.9  # <-- set your threshold here (e.g. optimal_threshold)

# ----------------------------
# Load cache index + scaler
# ----------------------------
with open(index_path, "rb") as f:
    index = pickle.load(f)

with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

# ----------------------------
# Load model bundle (booster + best_ntree)
# ----------------------------
bundle = joblib.load("best_model_xgb.pkl")
booster = bundle["booster"]
best_ntree = int(bundle["best_ntree"])

def predict_probs_booster(booster: xgb.Booster, X: np.ndarray, best_ntree: int) -> np.ndarray:
    """Predict probabilities from a native xgboost Booster (version-safe)."""
    dmat = xgb.DMatrix(X)
    try:
        probs = booster.predict(dmat, iteration_range=(0, best_ntree))
    except TypeError:
        probs = booster.predict(dmat, ntree_limit=best_ntree)
    return probs.astype(np.float32)

# ----------------------------
# Build TEST predictions dataframe WITH DATES + TICKER
# ----------------------------
rows = []

for entry in index:
    data = np.load(entry["cache_file"])
    ticker = entry["ticker"]

    X_full = data["X"].astype(np.float32)
    y_full = data["y"].astype(np.float32)
    dates  = pd.to_datetime(data["dates"])

    test_start = int(entry["test_start"])  # <-- TEST split boundary

    # TEST slice
    Xt = X_full[test_start:]
    yt = y_full[test_start:]
    dt = dates[test_start:]

    if len(Xt) == 0:
        continue

    # Scale like training pipeline
    Xt_scaled = scaler.transform(Xt).astype(np.float32, copy=False)

    # Predict
    pt = predict_probs_booster(booster, Xt_scaled, best_ntree)

    rows.append(pd.DataFrame({
        "Date": dt,
        "ticker": ticker,
        "y_true": yt.astype(np.int8),
        "prob_buy": pt,
    }))

test_preds = pd.concat(rows, ignore_index=True)

# Normalize Date to pure date (no time)
test_preds["Date"] = pd.to_datetime(test_preds["Date"]).dt.normalize()

# ✅ Total number of TEST days (unique dates in test across all tickers)
num_test_days = int(test_preds["Date"].nunique())

# Decision at threshold
test_preds["pred_buy"] = (test_preds["prob_buy"] >= SELECTED_THRESHOLD).astype(np.int8)

# Consider only executed trades (pred_buy == 1)
trades = test_preds[test_preds["pred_buy"] == 1].copy()

# success = label == 1, fail = label == 0
trades["is_success"] = (trades["y_true"] == 1).astype(np.int8)
trades["is_fail"]    = (trades["y_true"] == 0).astype(np.int8)

# ----------------------------
# ✅ Trades-per-day table with success/fail
# ----------------------------
daily = trades.groupby("Date").agg(
    num_trades=("pred_buy", "size"),
    num_success=("is_success", "sum"),
    num_fail=("is_fail", "sum"),
).reset_index().sort_values("Date")

daily["pct_success"] = np.where(daily["num_trades"] > 0, 100.0 * daily["num_success"] / daily["num_trades"], np.nan)
daily["pct_fail"]    = np.where(daily["num_trades"] > 0, 100.0 * daily["num_fail"] / daily["num_trades"], np.nan)

# Clean column order for display
daily_table = daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]].copy()

display(daily_table)

print(f"\nThreshold: {SELECTED_THRESHOLD:.3f}")
print(f"Test days (unique dates): {num_test_days}")
print(f"Total BUY trades in test: {int(daily['num_trades'].sum())}")
print(f"Days with >=1 trade: {len(daily)}")
print(f"Overall P(success | BUY) on test @ thr={SELECTED_THRESHOLD:.3f}: "
      f"{100.0 * daily['num_success'].sum() / max(1, daily['num_trades'].sum()):.2f}%")

# ----------------------------
# Plot: stacked bars (green=success, red=fail)
# ----------------------------
x_labels = daily["Date"].dt.strftime("%Y-%m-%d").to_list()
x = np.arange(len(x_labels))

plt.figure(figsize=(14, 6))
plt.bar(x, daily["num_success"].to_numpy(), color="green", label="success (TP)")
plt.bar(x, daily["num_fail"].to_numpy(), bottom=daily["num_success"].to_numpy(), color="red", label="fail (FP)")

plt.xticks(x, x_labels, rotation=45, ha="right")
plt.ylabel("Number of BUY trades")
plt.title(f"Test BUY trades per day @ thr={SELECTED_THRESHOLD:.3f} (green=success, red=fail)")
plt.grid(True, alpha=0.2, axis="y")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# don't execute the next cells
raise KeyboardInterrupt

# Real Life Test #3 :: using it to make a decision

In [ ]:
# ============================================================
# Real Life Testing (#3): scan ALL tickers at START OF DAY with OPEN price
# and print BUY/NO BUY using XGBoost (native Booster bundle)
#
# SCENARIO: Use at market open when you have:
# - Today's opening price
# - All previous complete bars (yesterday's full OHLCV and earlier)
#
# REQUIREMENTS (already in your notebook):
# - TICKERS (list of tickers)
# - WINDOW (int)  e.g. 30
# - INTERVAL (str) e.g. "1d"
# - BUY_THRESHOLD (float) e.g. 0.7
# - best_model_xgb.pkl saved as {"booster": Booster, "best_ntree": int}
# - scaler.pkl exists at .feature_cache_forward_return_w{WINDOW}/scaler.pkl
# ============================================================

import time
import pickle
import numpy as np
import pandas as pd
import yfinance as yf
import joblib
import xgboost as xgb
from pathlib import Path
from zoneinfo import ZoneInfo

# ----------------------------
# CONFIG (using hyperparameters set at the top of the notebook)
# ----------------------------
PROB_THRESHOLD = BUY_THRESHOLD  # Use the BUY_THRESHOLD from config
LOOKBACK_DAYS  = max(WINDOW * 3, 180)  # Ensure enough lookback for window + warmup
CHUNK_SIZE     = 40
SLEEP_BETWEEN_CHUNKS = 0.5
INTERVAL = "1d"

# Display current hyperparameters being used
print(f"Using hyperparameters:")
print(f"  BUY_THRESHOLD:      {BUY_THRESHOLD}")
print(f"  PROFIT_THRESHOLD:   {PROFIT_THRESHOLD}")
print(f"  HORIZON_BARS:       {HORIZON_BARS}")
print(f"  WINDOW:             {WINDOW}")
print(f"  INTERVAL:           {INTERVAL}")
print(f"  LOOKBACK_DAYS:      {LOOKBACK_DAYS}")
print()

tz_market = ZoneInfo("America/New_York")

# For start-of-day trading: use today's date at market open time
# This ensures we get all previous complete bars (yesterday and before)
today_market = pd.Timestamp.now(tz_market).normalize()  # Today at 00:00
start_market  = today_market - pd.Timedelta(days=LOOKBACK_DAYS)
# IMPORTANT: To get data from today, you have to set end date to tomorrow!
end_market    = today_market + pd.Timedelta(days=1)  # Include today's data


def _to_naive_utc(ts: pd.Timestamp) -> pd.Timestamp:
    return ts.tz_convert("UTC").tz_localize(None) if ts.tz is not None else ts


start_naive_utc = _to_naive_utc(start_market)
end_naive_utc   = _to_naive_utc(end_market)


def predict_probs_booster(booster: xgb.Booster, X: np.ndarray, best_ntree: int) -> np.ndarray:
    """Predict probabilities from a native xgboost Booster (version-safe)."""
    dmat = xgb.DMatrix(X)
    try:
        probs = booster.predict(dmat, iteration_range=(0, best_ntree))
    except TypeError:
        probs = booster.predict(dmat, ntree_limit=best_ntree)
    return probs.astype(np.float32)


def _ema(s: pd.Series, span: int) -> pd.Series:
    return s.ewm(span=span, adjust=False, min_periods=span).mean()


def _rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = (-delta).clip(lower=0.0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def _true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    tr1 = (high - low).abs()
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    return pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)


def _atr(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    tr = _true_range(high, low, close)
    return tr.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()


def _macd(close: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    ema_fast = _ema(close, fast)
    ema_slow = _ema(close, slow)
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False, min_periods=signal).mean()
    macd_hist = macd - macd_signal
    return macd, macd_signal, macd_hist


def _stoch_k(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    ll = low.rolling(period, min_periods=period).min()
    hh = high.rolling(period, min_periods=period).max()
    denom = (hh - ll).replace(0.0, np.nan)
    return 100.0 * (close - ll) / denom


def _stoch_d(stoch_k: pd.Series, smooth: int = 3) -> pd.Series:
    return stoch_k.rolling(smooth, min_periods=smooth).mean()


def _cci(high: pd.Series, low: pd.Series, close: pd.Series, n: int = 20) -> pd.Series:
    tp = (high + low + close) / 3.0
    sma = tp.rolling(n, min_periods=n).mean()
    md = tp.rolling(n, min_periods=n).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    return (tp - sma) / (0.015 * (md + 1e-12))


def _adx_dmi(high: pd.Series, low: pd.Series, close: pd.Series, n: int = 14):
    up_move   = high.diff()
    down_move = -low.diff()

    plus_dm  = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)

    prev_close = close.shift(1)
    tr = pd.concat(
    [
        (high - low).abs(),
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ],
    axis=1,
    ).max(axis=1)

    tr_smooth = tr.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    plus_dm_smooth  = pd.Series(plus_dm, index=high.index).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    minus_dm_smooth = pd.Series(minus_dm, index=high.index).ewm(alpha=1/n, adjust=False, min_periods=n).mean()

    plus_di  = 100.0 * (plus_dm_smooth  / (tr_smooth + 1e-12))
    minus_di = 100.0 * (minus_dm_smooth / (tr_smooth + 1e-12))

    dx  = 100.0 * ((plus_di - minus_di).abs() / ((plus_di + minus_di) + 1e-12))
    adx = dx.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    return adx, plus_di, minus_di


def add_my_ta_features(df: pd.DataFrame) -> pd.DataFrame:
    """Trend/EMA/MACD/RSI/Stoch/CCI/ADX (causal)"""
    out = df.copy()
    c = out["Close"]
    h = out["High"]
    l = out["Low"]

    out["ema_10"] = _ema(c, 10)
    out["ema_20"] = _ema(c, 20)
    out["ema_50"] = _ema(c, 50)

    macd, macd_sig, macd_hist = _macd(c, 12, 26, 9)
    out["macd"] = macd
    out["macd_signal"] = macd_sig
    out["macd_hist"] = macd_hist

    out["rsi_14"] = _rsi(c, 14)

    stoch_k = _stoch_k(h, l, c, 14)
    out["stoch_k_14"] = stoch_k
    out["stoch_d_14"] = _stoch_d(stoch_k, 3)

    out["cci_20"] = _cci(h, l, c, 20)

    adx, plus_di, minus_di = _adx_dmi(h, l, c, 14)
    out["adx_14"] = adx
    out["plus_di_14"] = plus_di
    out["minus_di_14"] = minus_di

    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def build_model_features_from_df(df_upto: pd.DataFrame, window: int) -> pd.DataFrame:
    """
    Build the SAME engineered feature set as training (no labels).
    
    SCENARIO: We BUY at market OPEN, only knowing today's OPEN price.
    Features use only PREVIOUS day's complete OHLCV + today's OPEN.
    
    FOR PRODUCTION: Pass in historical data with complete bars PLUS today's row
    with ONLY Open filled (High, Low, Close, Volume can be NaN or last known values).
    
    Expects columns: Date, Open, High, Low, Close, Volume.
    Returns DataFrame with columns ["Date", <features + lags>].
    """
    if "Date" not in df_upto.columns:
        raise ValueError("build_model_features_from_df expects a 'Date' column.")

    df = df_upto.copy()

    # Normalize and sort by Date
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    # Ensure numeric OHLCV
    for c in ["Open", "High", "Low", "Close", "Volume"]:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in df_upto for feature building.")
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # For production: last row might only have Open filled
    # We need at least Open for all rows
    df = df.dropna(subset=["Open"]).reset_index(drop=True)

    if len(df) < max(window + 2, 2):
        return pd.DataFrame(columns=["Date"])

    # =========================
    # Features: Only use data available at market OPEN
    # - All historical (t-1, t-2, ...) complete OHLCV
    # - Current day (t): ONLY the Open price (no High, Low, Close, Volume)
    # =========================
    
    # SHIFT all OHLCV data by 1 period to represent "previous" complete bars
    # The current row now represents what we knew BEFORE today opened
    df["prev_Open"] = df["Open"].shift(1)
    df["prev_High"] = df["High"].shift(1)
    df["prev_Low"] = df["Low"].shift(1)
    df["prev_Close"] = df["Close"].shift(1)
    df["prev_Volume"] = df["Volume"].shift(1)
    
    # Today's open is known (no shift needed for features that use it)
    df["today_Open"] = df["Open"]
    
    # All calculations below use PREVIOUS day's OHLCV (shifted data)
    # This ensures we only use information available before today's open
    
    # Basic per-bar returns / range (using PREVIOUS close)
    df["logret_1"] = np.log(df["prev_Close"] / df["prev_Close"].shift(1))
    df["ret_co"] = (df["prev_Close"] / df["prev_Open"]) - 1.0
    df["range_hl"] = (df["prev_High"] - df["prev_Low"]) / df["prev_Close"]

    # Gap from previous close to today's open
    df["gap_prevclose_to_open"] = (df["today_Open"] / df["prev_Close"]) - 1.0
    
    # Relative previous OHLC to close before that (scale-free price levels)
    close_t2 = df["prev_Close"].shift(1)  # Close from t-2
    df["open_rel_prevclose"] = (df["prev_Open"] / close_t2) - 1.0
    df["high_rel_prevclose"] = (df["prev_High"] / close_t2) - 1.0
    df["low_rel_prevclose"] = (df["prev_Low"] / close_t2) - 1.0

    # Candle anatomy of PREVIOUS bar (normalized)
    body = df["prev_Close"] - df["prev_Open"]
    denom_close = df["prev_Close"].replace(0.0, np.nan)
    df["body_size"] = body / denom_close
    upper_raw = df["prev_High"] - pd.concat([df["prev_Open"], df["prev_Close"]], axis=1).max(axis=1)
    lower_raw = pd.concat([df["prev_Open"], df["prev_Close"]], axis=1).min(axis=1) - df["prev_Low"]
    df["upper_wick_norm"] = upper_raw / denom_close
    df["lower_wick_norm"] = lower_raw / denom_close
    range_raw = (df["prev_High"] - df["prev_Low"]).replace(0.0, np.nan)
    df["body_to_range"] = body.abs() / range_raw
    df["clv"] = ((df["prev_Close"] - df["prev_Low"]) - (df["prev_High"] - df["prev_Close"])) / range_raw

    # ATR-based volatility (using PREVIOUS bars)
    df["atr14"] = _atr(df["prev_High"], df["prev_Low"], df["prev_Close"], period=14)
    df["atr14_norm"] = df["atr14"] / denom_close
    df["atr14_impulse_60"] = df["atr14_norm"] / df["atr14_norm"].rolling(60, min_periods=60).mean()

    # Rolling returns / momentum (using PREVIOUS close prices)
    for win in (3, 6, 12):
        roll = df["logret_1"].rolling(win, min_periods=win).sum()
        df[f"logret_sum_{win}"] = roll
        df[f"logret_mean_{win}"] = roll / float(win)

    # Simple N-bar price slopes (using PREVIOUS close)
    for win in (3, 6, 12):
        df[f"slope_close_{win}"] = (df["prev_Close"] - df["prev_Close"].shift(win)) / float(win)

    # Distance to rolling highs/lows (using PREVIOUS close)
    roll_max_40 = df["prev_Close"].rolling(40, min_periods=40).max()
    roll_min_20 = df["prev_Close"].rolling(20, min_periods=20).min()
    df["dist_to_HH_40"] = (df["prev_Close"] / roll_max_40) - 1.0
    df["dist_to_LL_20"] = (df["prev_Close"] / roll_min_20) - 1.0

    # Realized volatility (std of log returns from PREVIOUS closes)
    for win in (10, 20, 40):
        df[f"rv_{win}"] = df["logret_1"].rolling(win, min_periods=win).std()

    # Vol-of-vol
    df["rv_10_vol_20"] = df["rv_10"].rolling(20, min_periods=20).std()

    # Range expansion z-score (using PREVIOUS bars)
    re_mean = df["range_hl"].rolling(20, min_periods=20).mean()
    re_std = df["range_hl"].rolling(20, min_periods=20).std()
    df["range_expansion_z"] = (df["range_hl"] - re_mean) / re_std

    # Range position (using PREVIOUS bars)
    for win in (10, 20):
        roll_min_l = df["prev_Low"].rolling(win, min_periods=win).min()
        roll_max_h = df["prev_High"].rolling(win, min_periods=win).max()
        denom_range = (roll_max_h - roll_min_l).replace(0.0, np.nan)
        df[f"range_pos_{win}"] = (df["prev_Close"] - roll_min_l) / denom_range

    # Z-scores (using PREVIOUS close)
    close_mean_20 = df["prev_Close"].rolling(20, min_periods=20).mean()
    close_std_20 = df["prev_Close"].rolling(20, min_periods=20).std()
    df["z_close_20"] = (df["prev_Close"] - close_mean_20) / close_std_20

    logret_mean_20 = df["logret_1"].rolling(20, min_periods=20).mean()
    logret_std_20 = df["logret_1"].rolling(20, min_periods=20).std()
    df["z_logret_1_20"] = (df["logret_1"] - logret_mean_20) / logret_std_20

    # Parkinson volatility (using PREVIOUS bars)
    hl_ratio = (df["prev_High"] / df["prev_Low"]).replace({0.0: np.nan})
    parkinson_bar = (np.log(hl_ratio)) ** 2
    parkinson_const = 1.0 / (4.0 * np.log(2.0))
    df["parkinson_20"] = (parkinson_const * parkinson_bar.rolling(20, min_periods=20).mean()) ** 0.5

    # Garman-Klass volatility (using PREVIOUS bars)
    log_hl = np.log((df["prev_High"] / df["prev_Low"]).replace({0.0: np.nan}))
    log_co = np.log((df["prev_Close"] / df["prev_Open"]).replace({0.0: np.nan}))
    gk_var = 0.5 * (log_hl ** 2) - (2.0 * np.log(2.0) - 1.0) * (log_co ** 2)
    df["gk_vol_20"] = (gk_var.rolling(20, min_periods=20).mean()) ** 0.5

    # Volume features (using PREVIOUS bars only)
    df["dollar_vol_log"] = np.log1p(df["prev_Close"] * df["prev_Volume"])
    for win in (10, 20, 40):
        vol_ma = df["prev_Volume"].rolling(win, min_periods=win).mean()
        df[f"vol_rel_{win}"] = df["prev_Volume"] / vol_ma

    df["pv_agree_20"] = df["logret_1"] * df["vol_rel_20"]

    # OBV-like (using PREVIOUS close)
    sign_close = np.sign(df["prev_Close"].diff().fillna(0.0))
    df["obv"] = (sign_close * df["prev_Volume"]).cumsum()
    df["obv_change_20"] = df["obv"] - df["obv"].shift(20)

    # Create a dataframe with PREVIOUS complete bars for TA indicators
    df_prev = pd.DataFrame({
        "High": df["prev_High"],
        "Low": df["prev_Low"],
        "Close": df["prev_Close"],
        "Open": df["prev_Open"]
    })
    
    # Trend filters from EMAs / MACD (using PREVIOUS closes)
    df_prev = add_my_ta_features(df_prev)
    for col in df_prev.columns:
        if col not in ["High", "Low", "Close", "Open"]:
            df[col] = df_prev[col]

    for span in (10, 20, 50):
        ema_col = f"ema_{span}"
        df[f"dist_to_ema_{span}"] = (df["prev_Close"] / df[ema_col]) - 1.0
        df[f"ema_slope_{span}"] = df[ema_col] - df[ema_col].shift(1)

    # Interaction features
    df["trend_vol_20"] = df["dist_to_ema_20"] * df["rv_20"]
    df["mom_vol_6_20"] = df["logret_sum_6"] * df["vol_rel_20"]
    df["meanrev_vol_20"] = df["z_close_20"] / (df["rv_20"] + 1e-12)

    # VWAP 20 (using PREVIOUS bars only)
    tp = (df["prev_High"] + df["prev_Low"] + df["prev_Close"]) / 3.0
    vol_roll_20 = df["prev_Volume"].rolling(20, min_periods=20).sum()
    vwap_num_20 = (tp * df["prev_Volume"]).rolling(20, min_periods=20).sum()
    df["vwap_20"] = (vwap_num_20 / vol_roll_20).replace([np.inf, -np.inf], np.nan)
    df["dist_to_vwap_20"] = (df["prev_Close"] / df["vwap_20"]) - 1.0

    feat_now = [
    # Relative OHLC / gap (core for open trading)
    "open_rel_prevclose","high_rel_prevclose","low_rel_prevclose",
    "gap_prevclose_to_open",

    # Returns / range
    "logret_1","ret_co","range_hl",
    "logret_sum_3","logret_sum_6","logret_sum_12",
    # (often better than logret_mean_*; can drop means)
    "z_logret_1_20",

    # Candle anatomy
    "body_size","upper_wick_norm","lower_wick_norm","body_to_range","clv",

    # Volatility / risk regime (your strongest block)
    "atr14_norm","atr14_impulse_60",
    "gk_vol_20",
    "rv_10","rv_20","rv_40","rv_10_vol_20",
    "range_expansion_z",

    # Location in range / mean reversion
    "range_pos_10","range_pos_20","z_close_20",
    "dist_to_HH_40","dist_to_LL_20",

    # Volume confirmation
    "dollar_vol_log","vol_rel_10","vol_rel_20","vol_rel_40","pv_agree_20",
    "obv","obv_change_20",

    # Oscillators / trend (keep only the ones that tend to matter)
    "macd_hist","rsi_14","adx_14","cci_20","stoch_k_14","stoch_d_14",

    # Interactions you already had (2 of them were useful)
    "mom_vol_6_20","meanrev_vol_20",
    ]

    # Lag features for WINDOW (lags 1..window-1)
    lagged = [df[feat_now].shift(lag).add_suffix(f"_lag_{lag}") for lag in range(1, window)]
    Xdf = pd.concat([df[feat_now]] + lagged, axis=1)

    # Note: Volume features are already log-transformed in dollar_vol_log
    # No need for additional log1p transform

    Xdf = Xdf.replace([np.inf, -np.inf], np.nan)

    out = pd.concat([df[["Date"]], Xdf], axis=1)
    out = out.dropna(axis=0).reset_index(drop=True)
    return out


# ============================================================
# LOAD SCALER + XGB BOOSTER BUNDLE
# ============================================================
scaler_path = Path(f".feature_cache_forward_return_w{WINDOW}") / "scaler.pkl"
model_path  = Path("best_model_xgb.pkl")

assert scaler_path.exists(), f"Missing scaler at: {scaler_path}"
assert model_path.exists(),  f"Missing XGBoost model bundle at: {model_path}"

with open(scaler_path, "rb") as f:
    scaler_loaded = pickle.load(f)

bundle = joblib.load(model_path)
assert isinstance(bundle, dict) and "booster" in bundle and "best_ntree" in bundle, \
    "best_model_xgb.pkl must be a dict bundle: {'booster': Booster, 'best_ntree': int}"

booster = bundle["booster"]
best_ntree = int(bundle["best_ntree"])

# de-dup tickers while preserving order
seen = set()
TICKERS_UNIQ = []
for t in TICKERS:
    if t not in seen:
        TICKERS_UNIQ.append(t)
        seen.add(t)

def _extract_one_ticker_df(df_all: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Extract a single-ticker OHLCV frame from yf.download multi-ticker output."""
    if df_all is None or df_all.empty:
        return pd.DataFrame()

    cols = df_all.columns

    # common: df_all is MultiIndex with [Ticker, Field] or [Field, Ticker]
    if isinstance(cols, pd.MultiIndex) and ticker in cols.get_level_values(0):
        d = df_all[ticker].copy()
        need = ["Open", "High", "Low", "Close", "Volume"]
        if not set(need).issubset(d.columns):
            return pd.DataFrame()
        return d[need].copy()

    if isinstance(cols, pd.MultiIndex) and ticker in cols.get_level_values(-1):
        need = ["Open", "High", "Low", "Close", "Volume"]
        out = pd.DataFrame(index=df_all.index)
        for fld in need:
            if (fld, ticker) in cols:
                out[fld] = df_all[(fld, ticker)]
            elif (ticker, fld) in cols:
                out[fld] = df_all[(ticker, fld)]
            else:
                return pd.DataFrame()
        return out

    # single ticker case
    if not isinstance(cols, pd.MultiIndex):
        need = ["Open", "High", "Low", "Close", "Volume"]
        if set(need).issubset(df_all.columns):
            return df_all[need].copy()

    return pd.DataFrame()

def predict_prob_buy_from_history(df_upto: pd.DataFrame, today_open_price: float = None) -> dict:
    """
    Compute BUY probability at START OF DAY using only opening price.
    
    Args:
        df_upto: Complete historical bars (yesterday and before)
        today_open_price: Today's opening price (if available)
    
    For start-of-day trading:
    - df_upto contains all COMPLETE bars up to yesterday
    - today_open_price is today's opening price
    - We create a partial row for "today" with only Open filled
    """
    if df_upto is None or df_upto.empty:
        return {"ok": False, "reason": "no history"}

    # Get the last complete bar (yesterday)
    last_complete_time = df_upto.index[-1]
    last_complete_row = df_upto.iloc[-1]

    # Prepare dataframe with historical data
    tmp = df_upto.reset_index().rename(columns={df_upto.index.name or "index": "Date"})
    
    # If we have today's opening price, add it as a partial row
    if today_open_price is not None:
        # Create today's row with only Open price (other fields will be NaN)
        today_date = last_complete_time + pd.Timedelta(days=1)
        today_row = pd.DataFrame({
            "Date": [today_date],
            "Open": [today_open_price],
            "High": [np.nan],  # Not yet known at market open
            "Low": [np.nan],   # Not yet known at market open
            "Close": [np.nan], # Not yet known at market open
            "Volume": [np.nan] # Not yet known at market open
        })
        tmp = pd.concat([tmp, today_row], ignore_index=True)
    
    feats = build_model_features_from_df(tmp, WINDOW)
    if feats.empty:
        return {"ok": False, "reason": "features NaN (insufficient warmup)"}

    last = feats.iloc[-1]
    X_row = last.drop(labels=["Date"]).to_numpy(dtype=np.float32, copy=False).reshape(1, -1)

    Xs = scaler_loaded.transform(X_row).astype(np.float32, copy=False)
    prob_buy = float(predict_probs_booster(booster, Xs, best_ntree)[0])

    return {
        "ok": True,
        "bar_used_market": last_complete_time,
        "prob_buy": prob_buy,
        "cutoff_open": float(last_complete_row["Open"]),
        "cutoff_high": float(last_complete_row["High"]),
        "cutoff_low": float(last_complete_row["Low"]),
        "cutoff_close": float(last_complete_row["Close"]),
        "cutoff_volume": float(last_complete_row["Volume"]),
        "today_open": today_open_price if today_open_price is not None else None,
    }

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

results = []
errors = []

print(f"Scanning {len(TICKERS_UNIQ)} tickers at START OF DAY")
print(f"Market timezone: {tz_market}, today: {today_market}")
print(f"Downloading history from {start_market} to {end_market} (market time) ...")
print(f"This will get ALL previous complete bars plus TODAY'S opening price\n")

for chunk in chunks(TICKERS_UNIQ, CHUNK_SIZE):
    try:
        df_all = yf.download(
            tickers=chunk,
            start=start_naive_utc,
            end=end_naive_utc,
            interval=INTERVAL,
            auto_adjust=False,
            progress=False,
            group_by="ticker",
            threads=True,
        )
    except Exception as e:
        for t in chunk:
            errors.append({"ticker": t, "error": f"download failed: {e}"})
        time.sleep(SLEEP_BETWEEN_CHUNKS)
        continue

    if df_all is None or df_all.empty:
        for t in chunk:
            errors.append({"ticker": t, "error": "empty download"})
        time.sleep(SLEEP_BETWEEN_CHUNKS)
        continue

    # yfinance daily index is often naive; keep it in market tz for the cutoff filter
    idx = pd.to_datetime(df_all.index, errors="coerce")
    df_all = df_all.copy()
    df_all.index = idx
    df_all = df_all[~df_all.index.isna()]

    if df_all.index.tz is None:
        df_all.index = df_all.index.tz_localize(tz_market)
    else:
        df_all.index = df_all.index.tz_convert(tz_market)

    for t in chunk:
        d = _extract_one_ticker_df(df_all, t)
        if d.empty:
            errors.append({"ticker": t, "error": "no usable OHLCV in download"})
            continue

        d = d.sort_index()
        
        # Split into complete bars (yesterday and before) and today's data
        # Complete bars: have all OHLCV data
        d_complete = d.dropna(subset=["Open", "High", "Low", "Close", "Volume"])
        
        # Check if we have today's opening price
        # Today's bar might be incomplete (only Open available)
        today_bars = d[d.index >= today_market]
        today_open_price = None
        
        if not today_bars.empty and not pd.isna(today_bars.iloc[0]["Open"]):
            # We have today's opening price!
            today_open_price = float(today_bars.iloc[0]["Open"])
        
        # Check if today's opening price is available
        if today_open_price is None:
            print(f"⚠️  WARNING: {t} - Today's opening price not found. Market may not be open yet or data is incomplete. Skipping.")
            errors.append({"ticker": t, "error": "today's opening price not available"})
            continue
        
        # Use only complete historical bars (yesterday and before)
        d_history = d_complete[d_complete.index < today_market]
        
        if d_history.empty:
            errors.append({"ticker": t, "error": "no complete historical bars"})
            continue

        res = predict_prob_buy_from_history(d_history, today_open_price)
        if not res["ok"]:
            errors.append({"ticker": t, "error": res["reason"]})
            continue

        p = res["prob_buy"]
        decision = "BUY" if p >= PROB_THRESHOLD else "NO BUY"

        result_data = {
            "ticker": t,
            "bar_used_market": res["bar_used_market"],
            "prob_buy": p,
            "prob_buy_%": 100.0 * p,
            "decision": decision,
            "yesterday_close": res["cutoff_close"],
        }
        
        # Add today's opening price if available
        if today_open_price is not None:
            result_data["today_open"] = today_open_price
            result_data["gap_%"] = 100.0 * (today_open_price / res["cutoff_close"] - 1.0)
        
        results.append(result_data)

    time.sleep(SLEEP_BETWEEN_CHUNKS)

df_res = pd.DataFrame(results).sort_values(["decision", "prob_buy"], ascending=[False, False]).reset_index(drop=True)
df_err = pd.DataFrame(errors)
if not df_err.empty and "ticker" in df_err.columns:
    df_err = df_err.sort_values("ticker").reset_index(drop=True)

print("\n================ RESULTS ================")
print(f"OK: {len(df_res)} tickers | Errors: {len(df_err)} tickers")
print(f"\nNOTE: This scan is designed for START OF DAY trading.")
print(f"      Features use YESTERDAY's complete bar + TODAY's opening price (if available).")
print(f"      If 'today_open' column is present, the gap% shows overnight gap.\n")
display(df_res)

if not df_err.empty:
    print("\n================ ERRORS (top 50) ================")
    display(df_err.head(50))

out_csv = f"scan_start_of_day_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_res.to_csv(out_csv, index=False)
print(f"\nSaved results to: {out_csv}")
